In [ ]:
from ultralytics import YOLO
import cv2

# =========================
# Load trained YOLOv8 classification model
# =========================
trained_model_path = "mudra_classification_project/yolov8_mudra_cls/weights/best.pt"
model = YOLO(trained_model_path)

# Mudra class names
mudra_names = [
    'Alapadmam', 'Anjali', 'Aralam', 'Ardhachandran', 'Ardhapathaka', 'Berunda', 'Bramaram', 'Chakra', 'Chandrakala', 'Chaturam', 'Garuda',
    'Hamsapaksha', 'Hamsasyam', 'Kangulam', 'Kapith', 'Kapotham', 'Karkatta', 'Kartariswastika', 'Katakamukha_1', 'Katakamukha_2',
    'Katakamukha_3', 'Katakavardhana', 'Katrimukha', 'Khatva', 'Kilaka', 'Kurma', 'Matsya', 'Mayura', 'Mrigasirsha', 'Mukulam', 'Mushti',
    'Nagabandha', 'Padmakosha', 'Pasha', 'Pathaka', 'Pushpaputa', 'Sakata', 'Samputa', 'Sarpasirsha', 'Shanka', 'Shivalinga',
    'Shukatundam', 'Sikharam', 'Simhamukham', 'Suchi', 'Swastikam', 'Tamarachudam', 'Tripathaka', 'Trishulam', 'Varaha'
]

# =========================
# Initialize webcam
# =========================
cap = cv2.VideoCapture(0)  # 0 = default camera
if not cap.isOpened():
    print("[ERROR] Could not open webcam")
    exit()

print("[INFO] Press 'q' to quit.")

# =========================
# Real-time mudra prediction loop
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        print("[WARN] Failed to grab frame")
        break

    # Predict mudra class for the frame
    results = model.predict(frame, save=False)

    # Get top predicted class
    pred_idx = results[0].probs.top1
    pred_label = mudra_names[pred_idx]
    pred_conf = results[0].probs[pred_idx].item()

    # Overlay prediction on frame
    cv2.putText(frame, f'{pred_label} ({pred_conf:.2f})', (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

    # Show the frame
    cv2.imshow("Mudra Classification", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


In [ ]:
from ultralytics import YOLO
import cv2
from collections import deque
import numpy as np

# =========================
# Load trained YOLOv8 classification model
# =========================
trained_model_path = "mudra_classification_project/yolov8_mudra_cls/weights/best.pt"
model = YOLO(trained_model_path)

# Mudra class names
mudra_names = [
    'Alapadmam', 'Anjali', 'Aralam', 'Ardhachandran', 'Ardhapathaka', 'Berunda', 'Bramaram', 'Chakra', 'Chandrakala', 'Chaturam', 'Garuda',
    'Hamsapaksha', 'Hamsasyam', 'Kangulam', 'Kapith', 'Kapotham', 'Karkatta', 'Kartariswastika', 'Katakamukha_1', 'Katakamukha_2',
    'Katakamukha_3', 'Katakavardhana', 'Katrimukha', 'Khatva', 'Kilaka', 'Kurma', 'Matsya', 'Mayura', 'Mrigasirsha', 'Mukulam', 'Mushti',
    'Nagabandha', 'Padmakosha', 'Pasha', 'Pathaka', 'Pushpaputa', 'Sakata', 'Samputa', 'Sarpasirsha', 'Shanka', 'Shivalinga',
    'Shukatundam', 'Sikharam', 'Simhamukham', 'Suchi', 'Swastikam', 'Tamarachudam', 'Tripathaka', 'Trishulam', 'Varaha'
]

# =========================
# Initialize webcam
# =========================
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open webcam")
    exit()

# =========================
# Parameters for smooth predictions
# =========================
prediction_history = deque(maxlen=5)  # Keep last 5 predictions

# Define ROI rectangle for hand (can be adjusted)
roi_x, roi_y, roi_w, roi_h = 200, 100, 400, 400  # x, y, width, height

print("[INFO] Press 'q' to quit.")

# =========================
# Main loop
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        print("[WARN] Cannot read frame")
        break

    # Draw ROI rectangle
    cv2.rectangle(frame, (roi_x, roi_y), (roi_x + roi_w, roi_y + roi_h), (255, 0, 0), 2)
    roi = frame[roi_y:roi_y + roi_h, roi_x:roi_x + roi_w]

    # Predict mudra on ROI
    results = model.predict(roi, save=False)
    pred_idx = results[0].probs.top1
    pred_label = mudra_names[pred_idx]
    pred_conf = results[0].probs[pred_idx].item()

    # Add to history for smoothing
    prediction_history.append((pred_label, pred_conf))

    # Take the most frequent prediction in the last N frames
    labels, confs = zip(*prediction_history)
    unique_labels, counts = np.unique(labels, return_counts=True)
    smoothed_label = unique_labels[np.argmax(counts)]
    smoothed_conf = np.mean([c for l, c in prediction_history if l == smoothed_label])

    # Overlay prediction
    cv2.putText(frame, f'{smoothed_label} ({smoothed_conf:.2f})', (roi_x, roi_y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

    # Show the live frame
    cv2.imshow("Live Mudra Detection", frame)

    # Quit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()
